# Fase 3 · Preparación y modelado de datos
**Proyecto de Business Intelligence · COVID-19 · Equipo 7**
Análisis de Datos · Ingeniería de Sistemas · ITM · 2026-2

**Dataset:** Our World in Data (OWID), COVID-19: `owid-covid-data.csv`.

- **H1:** el PIB per cápita influye en la mortalidad reportada por COVID-19.
- **H2:** vacunarse más rápido redujo las muertes.

In [33]:
# Importación de libre
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


#pd.set_option('display.max_columns', 40)
#pd.set_option("display.width", 160)


url = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"
salida = Path("datos_procesados")
salida.mkdir(exist_ok=True)

In [34]:
df_raw = pd.read_csv(url, low_memory=False, parse_dates=['date'])

print(f"Filas: {df_raw.shape[0]:,} | Columnas: {df_raw.shape[1]}")
print(f"Rango de fechas: {df_raw['date'].min().date()} -> {df_raw['date'].max().date()}")
print(f"Ubicaciones únicas: {df_raw['location'].nunique()}")

Filas: 429,435 | Columnas: 67
Rango de fechas: 2020-01-01 -> 2024-08-14
Ubicaciones únicas: 255


In [35]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 429435 entries, 0 to 429434
Data columns (total 67 columns):
 #   Column                                      Non-Null Count   Dtype         
---  ------                                      --------------   -----         
 0   iso_code                                    429435 non-null  object        
 1   continent                                   402910 non-null  object        
 2   location                                    429435 non-null  object        
 3   date                                        429435 non-null  datetime64[ns]
 4   total_cases                                 411804 non-null  float64       
 5   new_cases                                   410159 non-null  float64       
 6   new_cases_smoothed                          408929 non-null  float64       
 7   total_deaths                                411804 non-null  float64       
 8   new_deaths                                  410608 non-null  float64      

In [36]:
# Eliminación de agregados y subdivisiones
SUBNACIONALES = ['OWID_ENG', 'OWID_SCT', 'OWID_WLS', 'OWID_NIR']

agregados = sorted(df_raw.loc[df_raw['continent'].isna(), 'location'].unique())
mask_paises = df_raw['continent'].notna() & ~df_raw['iso_code'].isin(SUBNACIONALES)
df = df_raw[mask_paises].copy()   

print(f"Agregados eliminados ({len(agregados)}):", agregados)
print("Subnacionales eliminados: England, Scotland, Wales, Northern Ireland")
print(f"Países/territorios restantes: {df['iso_code'].nunique()} | Filas: {len(df):,}")

Agregados eliminados (12): ['Africa', 'Asia', 'Europe', 'European Union (27)', 'High-income countries', 'Low-income countries', 'Lower-middle-income countries', 'North America', 'Oceania', 'South America', 'Upper-middle-income countries', 'World']
Subnacionales eliminados: England, Scotland, Wales, Northern Ireland
Países/territorios restantes: 239 | Filas: 397,676


In [37]:
# Duplicados en la clave (país, fecha)
dups = df.duplicated(['iso_code', 'date'], keep=False)
print(f"Filas con clave duplicada: {dups.sum():,}")
print(df.loc[dups, 'location'].value_counts().to_string())

# Muestra de un duplicado para entender su naturaleza
ejemplo = df.loc[dups].sort_values(['iso_code', 'date']).head(4)
ejemplo[['location', 'date', 'total_cases', 'total_deaths', 'people_fully_vaccinated_per_hundred']]

Filas con clave duplicada: 2,816
location
East Timor       2028
Faroe Islands     788


,location,date,total_cases,total_deaths,people_fully_vaccinated_per_hundred
123277,Faroe Islands,2021-01-29,NaN,NaN,0.0
125735,Faroe Islands,2021-01-29,655.0,1.0,NaN
123278,Faroe Islands,2021-01-30,NaN,NaN,NaN
125736,Faroe Islands,2021-01-30,655.0,1.0,NaN


In [38]:
# No se utiliza drop_duplicates() porque no son filas exactas, por ende se deben agrupar primero por la clave y luego tomar la primera fila de cada grupo. Esto elimina los duplicados en la clave (país, fecha) y conserva la primera fila de cada grupo.
n_antes = len(df)
df = df.groupby(['iso_code', 'date'], as_index=False).first()

print(f"Filas: {n_antes:,} -> {len(df):,}")
print(f"Duplicados restantes: {df.duplicated(['iso_code', 'date']).sum()}")

Filas: 397,676 -> 396,268
Duplicados restantes: 0


In [39]:
# Validación de variables estructurales y tipos
# verificamos que población, PIB, edad, etc. tengan UN solo valor por país.

ESTATICAS = ['population', 'gdp_per_capita', 'median_age', 'aged_65_older', 'aged_70_older',
             'human_development_index', 'life_expectancy', 'population_density',
             'hospital_beds_per_thousand', 'diabetes_prevalence', 'cardiovasc_death_rate',
             'extreme_poverty']

max_valores_distintos = df.groupby('iso_code')[ESTATICAS].nunique().max()
print("Máx. valores distintos por país (debe ser <= 1):")
print(max_valores_distintos.to_string())
assert (max_valores_distintos <= 1).all(), "Hay variables estructurales que cambian en el tiempo"

# 'continent' tiene solo 6 valores -> category ahorra memoria y acelera los groupby.
df['continent'] = df['continent'].astype('category')
print("\nTipos OK -> date:", df['date'].dtype, "| continent:", df['continent'].dtype)

Máx. valores distintos por país (debe ser <= 1):
population                    1
gdp_per_capita                1
median_age                    1
aged_65_older                 1
aged_70_older                 1
human_development_index       1
life_expectancy               1
population_density            1
hospital_beds_per_thousand    1
diabetes_prevalence           1
cardiovasc_death_rate         1
extreme_poverty               1

Tipos OK -> date: datetime64[ns] | continent: category


In [40]:
ACUM_EPI = ['total_cases', 'total_deaths']
ACUM_VAC = ['people_vaccinated_per_hundred', 'people_fully_vaccinated_per_hundred',
            'total_boosters_per_hundred']
ACUM = ACUM_EPI + ACUM_VAC

# El orden por país y fecha es OBLIGATORIO
df = df.sort_values(['iso_code', 'date']).reset_index(drop=True)

nulos_antes = df[ACUM].isna().sum()

df[ACUM] = df.groupby('iso_code')[ACUM].ffill()
df[ACUM_EPI] = df[ACUM_EPI].fillna(0)
caidas = {c: int((df.groupby('iso_code')[c].diff() < 0).sum()) for c in ACUM}
df[ACUM] = df.groupby('iso_code')[ACUM].cummax()
sobre_100 = int((df[ACUM_VAC] > 100).any(axis=1).sum())
df[ACUM_VAC] = df[ACUM_VAC].clip(upper=100)


for tot, nuevo in [('total_cases', 'new_cases'), ('total_deaths', 'new_deaths')]:
    df[nuevo] = df.groupby('iso_code')[tot].diff().fillna(df[tot])

resumen = pd.DataFrame({'nulos_antes': nulos_antes,
                        'nulos_despues': df[ACUM].isna().sum(),
                        'caidas_corregidas': pd.Series(caidas)})
print(resumen.to_string())
print(f"\nFilas con % vacunación > 100 recortadas: {sobre_100:,}")


chk = df.groupby('iso_code').agg(suma=('new_cases', 'sum'), final=('total_cases', 'last'))
assert np.allclose(chk['suma'], chk['final']), "Inconsistencia entre nuevos y acumulado"
assert (df[['new_cases', 'new_deaths']] >= 0).all().all(), "Hay valores diarios negativos"
print("casos nuevos = casos acumulados en todos los países | sin valores negativos")

                                     nulos_antes  nulos_despues  caidas_corregidas
total_cases                                 4552              0                 19
total_deaths                                4552              0                 12
people_vaccinated_per_hundred             333255         122575                  0
people_fully_vaccinated_per_hundred       336337         134717                  0
total_boosters_per_hundred                358572         205492                  0

Filas con % vacunación > 100 recortadas: 7,496
casos nuevos = casos acumulados en todos los países | sin valores negativos


In [41]:
df['anio'] = df['date'].dt.year
df['mes'] = df['date'].dt.month
df['trimestre'] = df['date'].dt.quarter
df['anio_mes'] = df['date'].dt.to_period('M').astype(str)   # 'YYYY-MM' -> llave de la dimensión tiempo

df[['iso_code', 'date', 'anio', 'mes', 'trimestre', 'anio_mes']].head(3)

,iso_code,date,anio,mes,trimestre,anio_mes
0,ABW,2020-01-05,2020,1,1,2020-01
1,ABW,2020-01-06,2020,1,1,2020-01
2,ABW,2020-01-07,2020,1,1,2020-01


In [42]:
print(f"Filas:       {len(df_raw):,} -> {len(df):,}")
print(f"Ubicaciones: {df_raw['location'].nunique()} -> {df['iso_code'].nunique()} países/territorios")
print(f"Periodo:     {df['date'].min().date()} -> {df['date'].max().date()}")

Filas:       429,435 -> 396,268
Ubicaciones: 255 -> 239 países/territorios
Periodo:     2020-01-01 -> 2024-08-14


In [43]:
# Selección de variables de interés para el modelado
# Lista de candidatas agrupadas por rol en el análisis
CANDIDATAS = {
    'identificacion':  ['iso_code', 'location', 'continent', 'date', 'anio', 'mes', 'trimestre', 'anio_mes'],
    'epidemiologicas': ['total_cases', 'new_cases', 'total_deaths', 'new_deaths'],
    'vacunacion':      ACUM_VAC,
    'contexto':        ['population', 'gdp_per_capita', 'aged_65_older', 'median_age', 'life_expectancy',
                        'population_density', 'hospital_beds_per_thousand', 'diabetes_prevalence',
                        'cardiovasc_death_rate', 'human_development_index', 'aged_70_older',
                        'extreme_poverty'],
}


UMBRAL_COBERTURA = 0.70
evaluar = CANDIDATAS['vacunacion'] + CANDIDATAS['contexto']
cobertura = df.groupby('iso_code')[evaluar].count().gt(0).mean().sort_values(ascending=False)

tabla_cob = cobertura.to_frame('cobertura_paises')
tabla_cob['pasa'] = tabla_cob['cobertura_paises'] >= UMBRAL_COBERTURA
print(tabla_cob.to_string(formatters={'cobertura_paises': '{:.0%}'.format}))

                                    cobertura_paises   pasa
population                                      100%   True
life_expectancy                                  97%   True
people_vaccinated_per_hundred                    92%   True
people_fully_vaccinated_per_hundred              91%   True
population_density                               90%   True
diabetes_prevalence                              86%   True
total_boosters_per_hundred                       85%   True
median_age                                       84%   True
aged_70_older                                    83%   True
cardiovasc_death_rate                            82%   True
gdp_per_capita                                   82%   True
aged_65_older                                    80%   True
human_development_index                          79%   True
hospital_beds_per_thousand                       72%   True
extreme_poverty                                  52%  False


In [44]:
UMBRAL_CORR = 0.90
PRIORIDAD = ['population', 'gdp_per_capita', 'aged_65_older', 'median_age', 'life_expectancy',
             'population_density', 'hospital_beds_per_thousand', 'diabetes_prevalence',
             'cardiovasc_death_rate', 'human_development_index', 'aged_70_older', 'extreme_poverty']
contexto_ok = [v for v in PRIORIDAD if cobertura[v] >= UMBRAL_COBERTURA]

nivel_pais = df.groupby('iso_code')[contexto_ok].first()
corr = nivel_pais.corr(method='spearman').abs()   
aceptadas, descartes = [], []
for var in contexto_ok:
    choque = [a for a in aceptadas if corr.loc[var, a] > UMBRAL_CORR]
    if choque:
        descartes.append({'variable': var, 'redundante_con': choque[0], 'rho': round(corr.loc[var, choque[0]], 3)})
    else:
        aceptadas.append(var)

print("Descartadas por redundancia:")
print(pd.DataFrame(descartes).to_string(index=False))
print("\nDescartadas por cobertura:", [v for v in PRIORIDAD if v not in contexto_ok])

Descartadas por redundancia:
               variable redundante_con   rho
             median_age  aged_65_older 0.908
human_development_index gdp_per_capita 0.942
          aged_70_older  aged_65_older 0.994

Descartadas por cobertura: ['extreme_poverty']


In [45]:
# Conjunto final de variables --------------------------------------------------------------------------------------
VARIABLES_FINALES = (CANDIDATAS['identificacion'] + CANDIDATAS['epidemiologicas']
                     + CANDIDATAS['vacunacion'] + aceptadas)
df_sel = df[VARIABLES_FINALES].copy()

print(f"Columnas: {df.shape[1]} (67 originales + 4 de tiempo) -> {df_sel.shape[1]}")
print("Variables de contexto seleccionadas:", aceptadas)

Columnas: 71 (67 originales + 4 de tiempo) -> 23
Variables de contexto seleccionadas: ['population', 'gdp_per_capita', 'aged_65_older', 'life_expectancy', 'population_density', 'hospital_beds_per_thousand', 'diabetes_prevalence', 'cardiovasc_death_rate']


In [46]:
RENOMBRES = {
    'location': 'pais', 'continent': 'continente', 'date': 'fecha',
    'total_cases': 'casos_acumulados', 'new_cases': 'casos_nuevos',
    'total_deaths': 'muertes_acumuladas', 'new_deaths': 'muertes_nuevas',
    'people_vaccinated_per_hundred': 'pct_al_menos_una_dosis',
    'people_fully_vaccinated_per_hundred': 'pct_vacunacion_completa',
    'total_boosters_per_hundred': 'refuerzos_por_100',
    'population': 'poblacion', 'gdp_per_capita': 'pib_per_capita',
    'aged_65_older': 'pct_mayores_65', 'life_expectancy': 'esperanza_vida',
    'population_density': 'densidad_poblacion', 'hospital_beds_per_thousand': 'camas_por_1000',
    'diabetes_prevalence': 'prevalencia_diabetes', 'cardiovasc_death_rate': 'tasa_muerte_cardiovascular',
}
df_sel = df_sel.rename(columns=RENOMBRES)
df_sel.columns.tolist()

['iso_code',
 'pais',
 'continente',
 'fecha',
 'anio',
 'mes',
 'trimestre',
 'anio_mes',
 'casos_acumulados',
 'casos_nuevos',
 'muertes_acumuladas',
 'muertes_nuevas',
 'pct_al_menos_una_dosis',
 'pct_vacunacion_completa',
 'refuerzos_por_100',
 'poblacion',
 'pib_per_capita',
 'pct_mayores_65',
 'esperanza_vida',
 'densidad_poblacion',
 'camas_por_1000',
 'prevalencia_diabetes',
 'tasa_muerte_cardiovascular']

In [47]:
paises = df_sel.groupby('iso_code').agg(
    pais=('pais', 'first'), continente=('continente', 'first'),
    poblacion=('poblacion', 'first'), pib_per_capita=('pib_per_capita', 'first'),
    pct_mayores_65=('pct_mayores_65', 'first'),
    pct_vacunacion_completa=('pct_vacunacion_completa', 'last'),   # último valor = cobertura alcanzada
)

# pd.qcut -> cortes por cuantiles; los países sin dato quedan como 'Sin dato' (no se inventan valores)
paises['grupo_ingreso'] = pd.qcut(paises['pib_per_capita'], 4,
                                  labels=['Q1 Bajo', 'Q2 Medio-bajo', 'Q3 Medio-alto', 'Q4 Alto'])
paises['grupo_edad'] = pd.qcut(paises['pct_mayores_65'], 3,
                               labels=['Joven', 'Intermedia', 'Envejecida'])
# pd.cut con cortes FIJOS (no cuantiles) porque 40 % y 70 % son umbrales con significado sanitario
paises['nivel_vacunacion'] = pd.cut(paises['pct_vacunacion_completa'], [-0.01, 40, 70, 100],
                                    labels=['Bajo (<40%)', 'Medio (40-70%)', 'Alto (>=70%)'])
paises['es_micro_estado'] = paises['poblacion'] < 100_000

for col in ['grupo_ingreso', 'grupo_edad', 'nivel_vacunacion']:
    paises[col] = paises[col].cat.add_categories('Sin dato').fillna('Sin dato')

for col in ['grupo_ingreso', 'grupo_edad', 'nivel_vacunacion', 'es_micro_estado']:
    print(paises[col].value_counts().sort_index().to_string(), '\n')

grupo_ingreso
Q1 Bajo          49
Q2 Medio-bajo    49
Q3 Medio-alto    48
Q4 Alto          49
Sin dato         44 

grupo_edad
Joven         64
Intermedia    64
Envejecida    64
Sin dato      47 

nivel_vacunacion
Bajo (<40%)       54
Medio (40-70%)    86
Alto (>=70%)      77
Sin dato          22 

es_micro_estado
False    201
True      38 



In [48]:
def calcular_metricas(t: pd.DataFrame) -> pd.DataFrame:
    t = t.copy()
    pob = t['poblacion']
    t['casos_por_millon'] = t['casos_acumulados'] / pob * 1e6
    t['muertes_por_millon'] = t['muertes_acumuladas'] / pob * 1e6
   
    t['letalidad_pct'] = np.where(t['casos_acumulados'] > 0,
                                  t['muertes_acumuladas'] / t['casos_acumulados'] * 100, np.nan)
    if 'casos_nuevos' in t.columns:
        t['incidencia_100k'] = t['casos_nuevos'] / pob * 1e5
        t['mortalidad_mes_por_millon'] = t['muertes_nuevas'] / pob * 1e6
    return t

# Prueba rápida de la función con un caso conocido
prueba = calcular_metricas(pd.DataFrame({'poblacion': [1_000_000], 'casos_acumulados': [500],
                                         'muertes_acumuladas': [10]}))
assert prueba.loc[0, 'muertes_por_millon'] == 10 and prueba.loc[0, 'letalidad_pct'] == 2
print("✔ calcular_metricas validada con caso de prueba")

✔ calcular_metricas validada con caso de prueba


In [49]:
# Atributos estáticos del país (seleccionados en 3.2) + dimensiones derivadas (3.3.1)
ATRIB_ESTATICOS = ['pais', 'continente', 'poblacion', 'pib_per_capita', 'pct_mayores_65', 'esperanza_vida',
                   'densidad_poblacion', 'camas_por_1000', 'prevalencia_diabetes', 'tasa_muerte_cardiovascular']
dim_pais = (df_sel.groupby('iso_code')[ATRIB_ESTATICOS].first()
            .join(paises[['grupo_ingreso', 'grupo_edad', 'nivel_vacunacion', 'es_micro_estado']])
            .reset_index())
dim_pais['continente'] = dim_pais['continente'].astype(str)
print(dim_pais.shape)
dim_pais.head(3)

(239, 15)


,iso_code,pais,continente,poblacion,pib_per_capita,pct_mayores_65,esperanza_vida,densidad_poblacion,camas_por_1000,prevalencia_diabetes,tasa_muerte_cardiovascular,grupo_ingreso,grupo_edad,nivel_vacunacion,es_micro_estado
0,ABW,Aruba,North America,106459,35973.78,13.08,76.29,584.80,NaN,11.62,NaN,Q4 Alto,Envejecida,Alto (>=70%),False
1,AFG,Afghanistan,Asia,41128772,1803.99,2.58,64.83,54.42,0.5,9.59,597.03,Q1 Bajo,Joven,Medio (40-70%),False
2,AGO,Angola,Africa,35588996,5819.50,2.40,61.15,23.89,NaN,3.94,276.04,Q2 Medio-bajo,Joven,Bajo (<40%),False


In [50]:
meses = pd.period_range(df_sel['fecha'].min(), df_sel['fecha'].max(), freq='M')
NOMBRES_MES = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
dim_tiempo = pd.DataFrame({
    'anio_mes': meses.astype(str),
    'anio': meses.year, 'mes': meses.month, 'trimestre': meses.quarter,
    'semestre': np.where(meses.month <= 6, 1, 2),
    'nombre_mes': [NOMBRES_MES[m - 1] for m in meses.month],
    'fecha_inicio_mes': meses.start_time.date,
})
print(dim_tiempo.shape)
dim_tiempo.head(3)

(56, 7)


,anio_mes,anio,mes,trimestre,semestre,nombre_mes,fecha_inicio_mes
0,2020-01,2020,1,1,1,Ene,2020-01-01
1,2020-02,2020,2,1,1,Feb,2020-02-01
2,2020-03,2020,3,1,1,Mar,2020-03-01


In [51]:
# Tabla de hechos mensual
# Agregación de diario a mensual.
fact = (df_sel.groupby(['iso_code', 'anio_mes'], as_index=False)
        .agg(casos_nuevos=('casos_nuevos', 'sum'),
             muertes_nuevas=('muertes_nuevas', 'sum'),
             casos_acumulados=('casos_acumulados', 'last'),
             muertes_acumuladas=('muertes_acumuladas', 'last'),
             pct_al_menos_una_dosis=('pct_al_menos_una_dosis', 'last'),
             pct_vacunacion_completa=('pct_vacunacion_completa', 'last'),
             refuerzos_por_100=('refuerzos_por_100', 'last')))

# La población se trae temporalmente desde dim_pais SOLO para calcular las tasas, luego se elimina:
# en un modelo estrella ese atributo ya vive en la dimensión (evita redundancia).
fact = (fact.merge(dim_pais[['iso_code', 'poblacion']], on='iso_code')
            .pipe(calcular_metricas)
            .drop(columns='poblacion'))
fact_covid_mensual = fact
print(fact_covid_mensual.shape)
fact_covid_mensual[fact_covid_mensual['iso_code'] == 'COL'].tail(3)

(13257, 14)


,iso_code,anio_mes,casos_nuevos,muertes_nuevas,casos_acumulados,muertes_acumuladas,pct_al_menos_una_dosis,pct_vacunacion_completa,refuerzos_por_100,casos_por_millon,muertes_por_millon,letalidad_pct,incidencia_100k,mortalidad_mes_por_millon
2517,COL,2024-06,1659.0,0.0,6387399.0,142727.0,82.92,71.72,28.81,123132.890317,2751.41541,2.234509,3.198132,0.0
2518,COL,2024-07,3501.0,0.0,6390900.0,142727.0,82.92,71.72,28.81,123200.380738,2751.41541,2.233285,6.749042,0.0
2519,COL,2024-08,976.0,0.0,6391876.0,142727.0,82.92,71.72,28.81,123219.195548,2751.41541,2.232944,1.881481,0.0


In [52]:
ultimo = fact_covid_mensual.sort_values('anio_mes').groupby('iso_code').last().reset_index()
resumen_pais = (dim_pais.merge(ultimo[['iso_code', 'anio_mes', 'casos_acumulados', 'muertes_acumuladas',
                                       'pct_al_menos_una_dosis', 'pct_vacunacion_completa', 'refuerzos_por_100']],
                               on='iso_code')
                .rename(columns={'anio_mes': 'ultimo_mes_reportado'})
                .pipe(calcular_metricas))
print(resumen_pais.shape)
resumen_pais.sort_values('muertes_por_millon', ascending=False)[
    ['pais', 'grupo_ingreso', 'nivel_vacunacion', 'muertes_por_millon', 'letalidad_pct']].head(5)

(239, 24)


,pais,grupo_ingreso,nivel_vacunacion,muertes_por_millon,letalidad_pct
168,Peru,Q2 Medio-bajo,Alto (>=70%),6489.799524,4.881293
20,Bulgaria,Q3 Medio-alto,Bajo (<40%),5706.319196,2.909801
23,Bosnia and Herzegovina,Q2 Medio-bajo,Bajo (<40%),5069.382378,4.060783
94,Hungary,Q3 Medio-alto,Medio (40-70%),4921.390980,2.198897
136,North Macedonia,Q3 Medio-alto,Medio (40-70%),4765.939723,2.843351


In [53]:
validaciones = {
    'PK única en dim_pais':                 dim_pais['iso_code'].is_unique,
    'PK única en dim_tiempo':               dim_tiempo['anio_mes'].is_unique,
    'PK compuesta única en fact':           not fact_covid_mensual.duplicated(['iso_code', 'anio_mes']).any(),
    'FK fact -> dim_pais íntegra':          fact_covid_mensual['iso_code'].isin(dim_pais['iso_code']).all(),
    'FK fact -> dim_tiempo íntegra':        fact_covid_mensual['anio_mes'].isin(dim_tiempo['anio_mes']).all(),
    'Σ casos mensuales = acumulado final':  np.isclose(fact_covid_mensual['casos_nuevos'].sum(),
                                                       resumen_pais['casos_acumulados'].sum()),
    'Σ muertes mensuales = acumulado final': np.isclose(fact_covid_mensual['muertes_nuevas'].sum(),
                                                        resumen_pais['muertes_acumuladas'].sum()),
    'Sin métricas negativas':               (fact_covid_mensual.select_dtypes('number').fillna(0) >= 0).all().all(),  # NaN = sin dato, no negativo
    'Vacunación dentro de [0, 100]':        fact_covid_mensual['pct_vacunacion_completa'].dropna().between(0, 100).all(),
}
for nombre, ok in validaciones.items():
    print(('✔' if ok else '✘'), nombre)
assert all(validaciones.values()), "El modelo NO pasó las validaciones"

print(f"\nMuertes totales en el modelo: {resumen_pais['muertes_acumuladas'].sum():,.0f}")
print(f"Casos totales en el modelo:   {resumen_pais['casos_acumulados'].sum():,.0f}")

✔ PK única en dim_pais
✔ PK única en dim_tiempo
✔ PK compuesta única en fact
✔ FK fact -> dim_pais íntegra
✔ FK fact -> dim_tiempo íntegra
✔ Σ casos mensuales = acumulado final
✔ Σ muertes mensuales = acumulado final
✔ Sin métricas negativas
✔ Vacunación dentro de [0, 100]

Muertes totales en el modelo: 7,058,885
Casos totales en el modelo:   775,900,191


In [54]:
## Exportación para la fase de implementación de KPIs
def construir_diccionario(tablas: dict) -> pd.DataFrame:
    # Genera el diccionario automáticamente a partir de las tablas reales (no se desactualiza)
    # y le añade una descripción legible de cada columna.
    DESCRIPCION = {
        'iso_code': 'Código ISO-3 del país (llave)', 'pais': 'Nombre del país', 'continente': 'Continente',
        'anio_mes': 'Periodo AAAA-MM (llave de tiempo)', 'anio': 'Año', 'mes': 'Mes (1-12)',
        'trimestre': 'Trimestre (1-4)', 'semestre': 'Semestre (1-2)', 'nombre_mes': 'Nombre corto del mes',
        'fecha_inicio_mes': 'Primer día del mes',
        'poblacion': 'Población total', 'pib_per_capita': 'PIB per cápita (USD PPA)',
        'pct_mayores_65': '% de población mayor de 65 años', 'esperanza_vida': 'Esperanza de vida al nacer (años)',
        'densidad_poblacion': 'Habitantes por km²', 'camas_por_1000': 'Camas hospitalarias por 1 000 hab.',
        'prevalencia_diabetes': '% de población 20-79 años con diabetes',
        'tasa_muerte_cardiovascular': 'Muertes cardiovasculares por 100 000 hab.',
        'grupo_ingreso': 'Cuartil de PIB per cápita (Q1 bajo - Q4 alto)',
        'grupo_edad': 'Tercil de % mayores de 65 (Joven / Intermedia / Envejecida)',
        'nivel_vacunacion': 'Cobertura final de vacunación completa (<40 / 40-70 / >=70 %)',
        'es_micro_estado': 'True si población < 100 000 (tasas inestables)',
        'casos_nuevos': 'Casos confirmados en el periodo (suma)', 'muertes_nuevas': 'Muertes confirmadas en el periodo (suma)',
        'casos_acumulados': 'Casos confirmados acumulados al cierre', 'muertes_acumuladas': 'Muertes acumuladas al cierre',
        'pct_al_menos_una_dosis': '% de población con al menos una dosis (cierre)',
        'pct_vacunacion_completa': '% de población con esquema completo (cierre)',
        'refuerzos_por_100': 'Dosis de refuerzo por cada 100 hab. (cierre)',
        'casos_por_millon': 'Casos acumulados por millón de hab.', 'muertes_por_millon': 'Muertes acumuladas por millón de hab.',
        'letalidad_pct': 'Muertes acumuladas / casos acumulados × 100',
        'incidencia_100k': 'Casos nuevos del mes por 100 000 hab.',
        'mortalidad_mes_por_millon': 'Muertes nuevas del mes por millón de hab.',
        'ultimo_mes_reportado': 'Último mes con datos del país',
    }
    filas = []
    for nombre, t in tablas.items():
        for col in t.columns:
            filas.append({'tabla': nombre, 'columna': col, 'tipo': str(t[col].dtype),
                          '%_nulos': round(t[col].isna().mean() * 100, 1),
                          'descripcion': DESCRIPCION.get(col, '')})
    return pd.DataFrame(filas)


def generar_datos_kpi(tablas: dict, ruta_salida: Path = salida) -> pd.DataFrame:
    # Exporta el modelo de datos de la Fase 3 para la fase de implementación de KPIs.
    #   tablas      -> {'nombre_archivo': DataFrame}
    #   ruta_salida -> carpeta destino (por defecto ../datos_procesados)
    #   retorna     -> DataFrame con archivo, filas, columnas y tamaño (trazabilidad)
    ruta_salida = Path(ruta_salida)
    ruta_salida.mkdir(parents=True, exist_ok=True)   # crea la carpeta si no existe; si existe, no falla

    tablas = {**tablas, 'diccionario_datos': construir_diccionario(tablas)}
    reporte = []
    for nombre, tabla in tablas.items():
        archivo = ruta_salida / f"{nombre}.csv"
        tabla.to_csv(archivo, index=False, encoding='utf-8-sig')
        reporte.append({'archivo': archivo.name, 'filas': len(tabla), 'columnas': tabla.shape[1],
                        'tamano_kb': round(archivo.stat().st_size / 1024, 1)})
    print(f"✔ {len(reporte)} archivos exportados en: {ruta_salida.resolve()}")
    return pd.DataFrame(reporte)


reporte_exportacion = generar_datos_kpi({
    'dim_pais': dim_pais,
    'dim_tiempo': dim_tiempo,
    'fact_covid_mensual': fact_covid_mensual,
    'resumen_pais': resumen_pais,
})
reporte_exportacion

✔ 5 archivos exportados en: D:\ITM\2026-2\Análisis de Datos\Entregables\Evaluación 3\code_covid\analisis-datos-evento2-grupo7\EntregaNo3\Evento3\fase3_preparacion_data\datos_procesados


,archivo,filas,columnas,tamano_kb
0,dim_pais.csv,239,15,25.2
1,dim_tiempo.csv,56,7,2.0
2,fact_covid_mensual.csv,13257,14,1636.4
3,resumen_pais.csv,239,24,47.0
4,diccionario_datos.csv,60,5,4.5
